## Scoring Algorithm

In [29]:
import pandas as pd

def recommend_studyroom(df, target_stations):
    # 지정된 역 근처의 스터디룸만 필터링
    df = df[df['subway'].isin(target_stations)].copy()
    
    # 전체 리뷰 수 계산 (방문자 리뷰 + 블로그 리뷰)
    df['visitor_review'] = df['visitor_review'].str.extract('(\d+)').astype(float).fillna(0)
    df['blog_review'] = df['blog_review'].str.extract('(\d+)').astype(float).fillna(0)
    df['total_reviews'] = df['visitor_review'] + df['blog_review']
    
    # 점수 계산을 위한 정규화 함수
    def normalize_score(series):
        if series.max() == series.min():
            return series.apply(lambda x: 1 if x > 0 else 0)
        return (series - series.min()) / (series.max() - series.min())
    
    # 역세권 점수 계산 (40%)
    df['distance'] = df['subway_distance'].str.extract(r'(\d+)(?=m)').astype(float).fillna(0)
    df['distance_score'] = normalize_score(1 - df['distance']) * 0.4

    # 리뷰 점수 계산 (30%)
    df['review_score'] = normalize_score(df['total_reviews']) * 0.3

    # 가격 점수 계산 (20%)
    df['price_score'] = df['orginized_price'].apply(lambda x: 0 if x == '가격정보 없음' else 1) * 0.2
    
    # 무선 인터넷 점수 계산 (5%)
    df['wifi_score'] = df['note'].str.contains('무선 인터넷', na=False).astype(float) * 0.05
    
    # 단체 이용 가능 점수 계산 (5%)
    df['group_score'] = df['note'].str.contains('단체 이용 가능', na=False).astype(float) * 0.05
    
    # 총점 계산
    df['total_score'] = df['review_score'] + df['distance_score'] + df['price_score'] + df['wifi_score'] + df['group_score']
    
    # 상위 5개 스터디룸 추천
    top_5 = df.nlargest(5, 'total_score')[['name', 'address', 'subway_distance', 'total_reviews', 'orginized_price']]
    top_5['total_reviews'] = top_5['total_reviews'].apply(lambda x: f'리뷰 수 {int(x)}개')

    return top_5

In [30]:
df = pd.read_csv('studyroom_df_result.csv')
recommendations = recommend_studyroom(df, ['선릉역', '한티역', '강남역'])

recommendations

,name,address,subway_distance,total_reviews,orginized_price
344,토즈 강남역토즈타워점,서울 강남구 강남대로84길 24-4 1~6층,강남역 3번출구에서 251m,리뷰 수 1200개,"2인 - 2시간 15,000원, 3~4인 - 2시간 30,000원, 5~6인 - 2..."
267,예인스페이스,서울 서초구 서초대로77길 9 4층,강남역 10번출구에서 66m,리뷰 수 304개,"60인실 - 1시간 110,000원, 32인실 - 1시간 66,000원, 7~12인..."
259,강남 스터디룸 망고 모임공간,서울 서초구 서초대로77길 9 누드죤빌딩 9층,강남역 10번출구에서 66m,리뷰 수 224개,"1인 - 1시간 2,500원, 키위룸 (24인) - 1시간 55,000원, 망고대형..."
0,토즈 선릉점,서울 강남구 테헤란로70길 14-8 세왕개발빌딩 9층,선릉역 1번출구에서 437m,리뷰 수 765개,"1~2인 - 1시간 7,500원, 3~4인 - 1시간 12,500원, 5~6인 - ..."
266,코지모임공간 강남역 5호점,서울 서초구 서초대로78길 38 호정빌딩 7층,강남역 5번출구에서 118m,리뷰 수 146개,"4인실 - 1시간 4,000원, 8인실 - 1시간 20,000원, 10인실 - 1시..."


## Filtering & Scoring

In [31]:
def recommend_studyroom(df, target_stations, wifi=True, group=True):
    # 지정된 역 근처의 스터디룸만 필터링
    df = df[df['subway'].isin(target_stations)].copy()
    
    # 무선 인터넷이 되는 스터디룸만 필터링
    if wifi:
        df = df[df['note'].str.contains('무선 인터넷', na=False)]
    
    # 단체 이용이 되는 스터디룸만 필터링
    if group:
        df = df[df['note'].str.contains('단체 이용 가능', na=False)]
    
    # 전체 리뷰 수 계산 (방문자 리뷰 + 블로그 리뷰)
    df['visitor_review'] = df['visitor_review'].str.extract('(\d+)').astype(float).fillna(0)
    df['blog_review'] = df['blog_review'].str.extract('(\d+)').astype(float).fillna(0)
    df['total_reviews'] = df['visitor_review'] + df['blog_review']
    
    # 점수 계산을 위한 정규화 함수
    def normalize_score(series):
        if series.max() == series.min():
            return series.apply(lambda x: 1 if x > 0 else 0)
        return (series - series.min()) / (series.max() - series.min())
    
    # 역세권 점수 계산 (40%)
    df['distance'] = df['subway_distance'].str.extract(r'(\d+)(?=m)').astype(float).fillna(0)
    df['distance_score'] = normalize_score(1 - df['distance']) * 0.4

    # 리뷰 점수 계산 (30%)
    df['review_score'] = normalize_score(df['total_reviews']) * 0.3

    # 가격 점수 계산 (30%)
    df['price_score'] = df['orginized_price'].apply(lambda x: 0 if x == '가격정보 없음' else 1) * 0.3
    
    # 총점 계산
    df['total_score'] = df['review_score'] + df['distance_score'] + df['price_score']
    
    # 상위 5개 스터디룸 추천
    top_5 = df.nlargest(5, 'total_score')[['name', 'address', 'subway_distance', 'total_reviews', 'orginized_price']]
    top_5['total_reviews'] = top_5['total_reviews'].apply(lambda x: f'리뷰 수 {int(x)}개')

    return top_5

In [32]:
df = pd.read_csv('studyroom_df_result.csv')
recommendations = recommend_studyroom(df, ['선릉역', '한티역', '강남역'])

recommendations

,name,address,subway_distance,total_reviews,orginized_price
344,토즈 강남역토즈타워점,서울 강남구 강남대로84길 24-4 1~6층,강남역 3번출구에서 251m,리뷰 수 1200개,"2인 - 2시간 15,000원, 3~4인 - 2시간 30,000원, 5~6인 - 2..."
267,예인스페이스,서울 서초구 서초대로77길 9 4층,강남역 10번출구에서 66m,리뷰 수 304개,"60인실 - 1시간 110,000원, 32인실 - 1시간 66,000원, 7~12인..."
259,강남 스터디룸 망고 모임공간,서울 서초구 서초대로77길 9 누드죤빌딩 9층,강남역 10번출구에서 66m,리뷰 수 224개,"1인 - 1시간 2,500원, 키위룸 (24인) - 1시간 55,000원, 망고대형..."
0,토즈 선릉점,서울 강남구 테헤란로70길 14-8 세왕개발빌딩 9층,선릉역 1번출구에서 437m,리뷰 수 765개,"1~2인 - 1시간 7,500원, 3~4인 - 1시간 12,500원, 5~6인 - ..."
266,코지모임공간 강남역 5호점,서울 서초구 서초대로78길 38 호정빌딩 7층,강남역 5번출구에서 118m,리뷰 수 146개,"4인실 - 1시간 4,000원, 8인실 - 1시간 20,000원, 10인실 - 1시..."
